# 데이터 불균형 

- 데이터 불균형으로 발생하는 골칫거리들 

## 소수의 클래스 과소 평가 

## 모델의 과적합 (Overfitting)

- 모델이 정상 데이터만 과도하게 학습하다 보면, 실제 세상에서 발생하는 드문 사기 거래를 제대로 인식하지 못함 

## 일반화 능력의 저하 

- 모델이 학습한 내용을 새로운, 보지 못한 데이터에도 잘 적용하는 능력이 저하 됨. 

## 오버샘플링을 통한 소수 클래스 증가시키기 

- 단순 무작위 복제
- SMOTE ( Synthetic Minorirty Over-sampling Technique )

```python
import pandas as pd

df = pd.read_csv('df.csv')

display(df.head())
display(df.info())
display(df['거래 유형'].unique())
```

# result 

```

거래 금액	거래 유형	계좌 나이	이전 거래와의 시간 간격	사기 여부
0	3949	입금	9	40	사기
1	8340	국내송금	15	11	사기
2	4870	해외송금	6	30	사기
3	704	출금	15	40	사기
4	2976	입금	17	31	사기
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   거래 금액          300 non-null    int64 
 1   거래 유형          300 non-null    object
 2   계좌 나이          300 non-null    int64 
 3   이전 거래와의 시간 간격  300 non-null    int64 
 4   사기 여부          300 non-null    object
dtypes: int64(3), object(2)
memory usage: 11.8+ KB
None
array(['입금', '국내송금', '해외송금', '출금'], dtype=object)
```

```python
from sklearn.model_selection import train_test_split

# 레이블 인코딩
df['거래 유형'] = df['거래 유형'].map({'출금':0, '입금':1, '국내송금':2, '해외송금':3})
df['사기 여부'] = df['사기 여부'].map({'사기': 1, '정상': 0})

X = df.drop('사기 여부', axis=1)
y = df['사기 여부']

# # 훈련 및 테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
display(df.head())
```

---

## Result 

```
	거래 금액	거래 유형	계좌 나이	이전 거래와의 시간 간격	사기 여부
0	3949	1	9	40	1
1	8340	2	15	11	1
2	4870	3	6	30	1
3	704	0	15	40	1
4	2976	1	17	31	1

```


```python
print("원본 데이터셋의 클래스 분포:\n", y_train.value_counts(), "\n")

print("원본 데이터셋의 클래스 분포:\n", y_train.value_counts(normalize=True) * 100)
```

```
원본 데이터셋의 클래스 분포:
 0    169
1     41
Name: 사기 여부, dtype: int64 

원본 데이터셋의 클래스 분포:
 0    80.47619
1    19.52381
Name: 사기 여부, dtype: float64
```

# 불균형 데이터를 균형있게 만드는 방법

언더샘플링(Undersampling) 은 다수 클래스의 샘플을 줄이는 방식으로, 희귀 사건들을 모두 선택하고 풍부한 사건들을 적절히 추출하여 데이터셋을 균일하게 만듭니다.
이는 큰 데이터셋에서 중요하지 않은 정보를 줄여주어 모델이 핵심 패턴에 더 집중할 수 있게 도와줍니다.

반면 오버샘플링(Oversampling) 은 소수 클래스의 샘플을 인위적으로 늘려, 모든 클래스가 비슷한 수의 샘플을 가지도록 합니다.
이 방식은 모델이 소수 클래스에 대해 더 많은 정보를 학습하도록 하여, 보다 균형 잡힌 예측을 가능하게 합니다.

언더샘플링과 오버샘플링은 각각의 장단점을 가지고 있으며, 상황에 따라 선택적으로 또는 함께 사용될 수 있습니다.
이제, 이러한 개념들을 바탕으로 오버샘플링에 대해 보다 심층적으로 알아보는 시간을 갖도록 하겠습니다.

## 오버샘플링 1️⃣: 단순 무작위 복제

- 소수 클래스의 샘플들을 단순히 복사하여 늘리는 접근 방식 을 취합니다.


```python
from imblearn.over_sampling import RandomOverSampler

# 오버샘플링 적용
ros = RandomOverSampler(random_state=42)
X_resampled, y_resampled = ros.fit_resample(X_train, y_train)

# 결과 출력
print("오버샘플링 적용 후의 클래스 분포:\n", y_resampled.value_counts())
```

---

```
버샘플링 적용 후의 클래스 분포:
 0    169
1    169
Name: 사기 여부, dtype: int64
```

## 오버샘플링 2️⃣: SMOTE 기법

- 오버 샘플링의 두 번째 방법으로,
SMOTE(Synthetic Minority Over-sampling Technique),   
즉 '합성 소수 오버샘플링 기법'은 데이터 불균형 문제를 해결하는 데 정말 유용한 도구입니다.  
SMOTE의 기본 목표는 소수 클래스의 오버샘플링을 통해 보다 균형 잡힌 데이터 세트를 만드는 것입니다.

이 기술은 기존 소수 클래스 인스턴스 간의 특성을 보간하여 새로운 합성 샘플을 생성 합니다.

1. 이웃 선택
SMOTE에서는 먼저 소수 클래스의 특정 인스턴스(데이터 포인트)를 선택합니다.  
그 다음, 이 인스턴스와 가장 가까이 있는 다른 소수 클래스 인스턴스들 중에서 하나 이상을 '이웃'으로 선택합니다.  
이 이웃들은 선택한 원본 인스턴스와 비슷한 특성을 가진 데이터 포인트들입니다.  

2. 임의의 이웃 선택
이 중에서 하나의 이웃을 무작위로 선택합니다.  
이 과정은 데이터셋에서 다양성을 보장하기 위해 무작위성을 도입하는 것입니다.  
선택된 이웃은 원본 인스턴스와 비슷하지만, 완전히 동일하지는 않습니다.  

3. 특성 차이 계산
선택된 이웃과 원본 인스턴스 간의 각 특성 값(예: 데이터 포인트의 각 변수 값)의 차이를 계산합니다.  
이 차이는 새로운 데이터 포인트를 생성하는 데 사용될 '간격' 또는 '벡터'를 나타냅니다.   

4. 새 인스턴스 생성
계산된 특성 차이에 0과 1 사이의 무작위 수(랜덤 스칼라 값)를 곱합니다.  
이것은 새로운 합성 인스턴스가 원본 인스턴스와 선택된 이웃 사이의 '중간점'에 위치하도록 합니다.  
무작위 수를 곱함으로써, 생성된 새 인스턴스는 원본 인스턴스와 정확히 동일하지 않고 약간의 변화를 가지게 됩니다.  

5. 합성 인스턴스 추가
이렇게 계산된 새 특성 값은 원본 인스턴스의 특성 값에 추가되어, 최종적으로 새로운 합성 인스턴스를 생성합니다.  
이 새로운 합성 인스턴스는 원본 데이터셋에 추가됩니다.  

이러한 방식으로 여러 소수 클래스 인스턴스에 대해 반복함으로써,  
SMOTE는 원래 데이터 세트에 추가할 수 있는 더 크고 다양한 합성 소수 클래스 인스턴스 세트를 만듭니다.  


### 예시 설명 😇

- 원본 데이터 포인트의 특성 값: 10. 
- 선택된 이웃의 특성 값: 20. 
- 두 포인트 사이의 차이(간격): 20 - 10 = 10. 

여기에 0.1을 곱하면 (0과 1 사이의 무작위 수):  

- 축소된 간격: 10 * 0.1 = 1. 
- 새로운 데이터 포인트의 특성 값: 10(원본) + 1 = 11. 

이 경우 새로운 데이터 포인트는 원본 데이터 포인트(10)와 매우 가까운 위치(11)에 생성됩니다.   
반대로, 0.9와 같이 더 큰 수를 곱하면 새로운 데이터 포인트는 선택된 이웃(20)에 더 가깝게 생성됩니다.  
중요한 점
SMOTE에서 이러한 방식으로 데이터 포인트를 생성하는 것은 데이터셋에 다양성을 추가하고,  
모델이 소수 클래스의 특징을 더 잘 학습하도록 돕기 위함입니다. 0과 1 사이의 무작위 수를 사용함으로써,   
새로운 데이터   포인트는 원본 데이터셋의 자연스러운 변형을 반영하며,  이는 과적합을 줄이는 데에도 도움이 됩니다.

# 언더 샘플링 

## 무작위 언더샘플링 (Random UnderSampling)  

가장 간단한 형태의 언더샘플링 방법입니다.  
이 방법은 다수 클래스의 데이터 포인트들을 무작위로 제거합니다.  
하지만 이 방식은 중요한 데이터를 잃어버릴 위험이 있습니다.  

## NearMiss  

다수 클래스의 데이터 포인트 중 소수 클래스의 데이터 포인트와 가장 가까운 것들만을 보존하는 방법입니다.  
이 방법은 데이터의 균형을 맞추면서도 소수 클래스에 유사한 다수 클래스의 데이터를 유지하려고 시도합니다.  

```python
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('df.csv')

# 레이블 인코딩
df['거래 유형'] = df['거래 유형'].map({'출금':0, '입금':1, '국내송금':2, '해외송금':3})
df['사기 여부'] = df['사기 여부'].map({'사기': 1, '정상': 0 })

X = df.drop('사기 여부', axis=1)
y = df['사기 여부']

# # 훈련 및 테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
from imblearn.under_sampling import RandomUnderSampler

# 언더샘플링 적용
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)

# 결과 출력
print("언더샘플링 적용 후의 클래스 분포:\n", y_under.value_counts())


```

```
언더샘플링 적용 후의 클래스 분포:
 0    41
1    41
Name: 사기 여부, dtype: int64
```

## 장점 👍
### 간단함과 접근성
- 복잡한 알고리즘이나 추가적인 데이터 처리가 필요 없어, 쉽게 구현하고 적용할 수 있습니다.  

### 계산 효율성
- 데이터를 줄이는 과정이 간단하기 때문에, 처리 시간과 계산 비용이 낮습니다.  

## 단점 👎
### 중요 정보 손실
무작위로 데이터를 제거하므로, 중요한 정보나 패턴을 잃어버릴 수 있습니다.  

## 데이터 대표성 저하
### 데이터셋에서 중요한 다양성이나 대표성이 줄어들 수 있어, 모델의 일반화 능력에 영향을 미칠 수 있습니다.

# NearMiss 기법

- NearMiss는 다수 클래스의 샘플 중 소수 클래스 샘플과 가장 가까운 것들을 선택하여 보존합니다.  
- NearMiss 방법에는 여러 버전이 있으며, 각각 다른 기준으로 가까운 샘플들을 선택합니다.  

- NearMiss(version=1)은 각 소수 클래스 샘플에 대해 가장 가까운 다수 클래스 샘플을 선택 합니다. 이 방법은 소수 클래스 샘플과 가장 가까운 다수 클래스 샘플을 유지하여 데이터의 균형을 맞춥니다.  
- NearMiss(version=2)는 다수 클래스 샘플 중에서 소수 클래스 샘플과 가장 가까운 평균 거리 를 가진 샘플을 선택합니다. 이 방법은 다수 클래스 내에서 소수 클래스 샘플과 전반적으로 가까운 샘플을 유지합니다.   
- NearMiss(version=3)는 소수 클래스 샘플의 이웃을 먼저 찾은 후, 이 이웃 중 다수 클래스 샘플을 선택 합니다. 이는 소수 클래스 샘플과 직접적으로 연관된 다수 클래스 샘플을 유지하는 방식입니다.  



```python
from imblearn.under_sampling import NearMiss

# NearMiss 버전별 적용
nm1 = NearMiss(version=1)
nm2 = NearMiss(version=2)
nm3 = NearMiss(version=3)

X_nm1, y_nm1 = nm1.fit_resample(X_train, y_train)
X_nm2, y_nm2 = nm2.fit_resample(X_train, y_train)
X_nm3, y_nm3 = nm3.fit_resample(X_train, y_train)

# 결과 출력
print("NearMiss-1 적용 후의 클래스 분포:\n", y_nm1.value_counts())
print("NearMiss-1 적용 후의 클래스 분포:\n", y_nm2.value_counts())
print("NearMiss-1 적용 후의 클래스 분포:\n", y_nm3.value_counts())
```


```
NearMiss-1 적용 후의 클래스 분포:
 0    41
1    41
Name: 사기 여부, dtype: int64
NearMiss-1 적용 후의 클래스 분포:
 0    41
1    41
Name: 사기 여부, dtype: int64
NearMiss-1 적용 후의 클래스 분포:
 0    41
1    41
Name: 사기 여부, dtype: int64
```

## 장점 👍
### 클래스 간 균형 개선 
- NearMiss는 다수 클래스의 샘플을 줄여 소수 클래스와의 균형을 개선합니다. 이는 특히 소수 클래스에 중요한 정보가 있는 경우 머신러닝 모델의 성능 향상에 도움이 됩니다.  

### 과적합 감소
- 다수 클래스의 지배적인 영향을 줄임으로써, 모델이 소수 클래스를 무시하고 다수 클래스에 과적합되는 것을 방지할 수 있습니다.  

### 노이즈 및 이상치 제거

- 특정 버전의 NearMiss는 다수 클래스 내의 노이즈와 이상치를 제거할 수 있으며, 이는 더 깨끗하고 일반화된 데이터셋을 생성하는 데 도움이 됩니다.  

## 단점 👎
### 정보 손실 위험 

- 다수 클래스에서 중요한 샘플을 제거할 위험이 있습니다. 이로 인해 모델이 중요한 정보를 놓칠 수 있으며, 전체적인 성능이 저하될 수 있습니다.

### 선택 기준의 한계

- NearMiss는 가까운 샘플을 기준으로 선택하기 때문에, 항상 가장 유용하거나 대표적인 샘플을 선택한다는 보장이 없습니다. 따라서, 때로는 중요하지 않은 샘플을 유지하고 중요한 샘플을 제거할 수도 있습니다.  

### 데이터 분포의 왜곡

- 특히 NearMiss-1과 같은 방법은 데이터의 원래 분포를 왜곡시킬 수 있으며, 이는 모델이 실제 세계 데이터에 적용될 때 문제를 일으킬 수 있습니다.  

### 계산 복잡성  

특히 NearMiss-3 같은 방법은 계산이 복잡하고 시간이 많이 소요될 수 있으며, 큰 데이터셋에서는 비효율적일 수 있습니다.  